<a href="https://colab.research.google.com/github/turkson-michael/Machine-learning-/blob/main/Battery_health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import math
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless-safe; change to "TkAgg" for interactive
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# Optional — uncomment if scipy is available (needed for real .mat loading)
from scipy.io import loadmat

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [3]:
EOL_THRESHOLD    = 1.4      # Ah — NASA standard end-of-life capacity
INITIAL_CAPACITY = 1.856    # Ah — nominal fresh capacity
WINDOW_SIZE      = 10       # cycles used per feature vector
TRAIN_BATTERY    = "B0005"
TEST_BATTERY     = "B0007"
EPOCHS           = 100
LEARNING_RATE    = 0.002
BATCH_SIZE       = 32
HIDDEN_SIZES     = [64, 32, 16]   # hidden layer neurons (input=9, output=1)
OUTPUT_DIR       = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
def generate_synthetic_nasa_data(battery_id: str = "B0005",
                                  n_cycles: int = 168) -> dict:
    """
    Synthesises degradation data that statistically mirrors NASA PCoE batteries.

    Capacity fade model:
        C(k) = C(k-1) - Δ_weibull(k) + ε_k
        Δ_weibull ~ Weibull-shaped fade rate
        ε_k       ~ N(0, σ²)    process noise

    Returns
    -------
    dict with keys:
        'battery_id', 'records' (list of dicts), 'initial_capacity', 'eol_threshold'
    """
    rng = np.random.RandomState(ord(battery_id[-1]))

    initial_cap = INITIAL_CAPACITY + rng.normal(0, 0.02)
    fade_rate   = 0.0025 + rng.uniform(0, 0.0008)
    weibull_k   = 2.8    + rng.uniform(0, 0.4)     # shape parameter

    capacity = initial_cap
    records  = []

    for c in range(1, n_cycles + 1):
        t = c / n_cycles
        weibull_fade = fade_rate * (t * n_cycles / 80) ** (weibull_k - 1)
        capacity    -= weibull_fade + rng.normal(0, 0.003)
        capacity     = max(capacity, EOL_THRESHOLD - 0.05)

        voltage     = 4.19 - 0.12 * t + rng.normal(0, 0.008)
        current     = -(2.0 + rng.normal(0, 0.05))
        temperature = 24.0 + 4.0 * t  + rng.normal(0, 0.5)
        soh         = float(np.clip((capacity - EOL_THRESHOLD) /
                                    (initial_cap - EOL_THRESHOLD), 0, 1))
        rul         = max(0, n_cycles - c)
        is_eol      = capacity <= EOL_THRESHOLD

        records.append({
            "cycle":       c,
            "capacity":    round(capacity,    4),
            "voltage":     round(voltage,     4),
            "current":     round(current,     4),
            "temperature": round(temperature, 2),
            "soh":         round(soh,         4),
            "rul":         rul,
            "is_eol":      is_eol,
        })

    return {
        "battery_id":        battery_id,
        "records":           records,
        "initial_capacity":  initial_cap,
        "eol_threshold":     EOL_THRESHOLD,
    }

In [5]:
def load_real_nasa_data(mat_path: str, battery_id: str) -> dict:
    """
    Loads a real NASA .mat file and extracts cycle-level features.
    Requires: scipy.io.loadmat

    Expected .mat structure (NASA format):
        B0005.cycle(i).type   = 'charge' | 'discharge' | 'impedance'
        B0005.cycle(i).data.Capacity    (discharge cycles only)
        B0005.cycle(i).data.Voltage_measured
        B0005.cycle(i).data.Current_measured
        B0005.cycle(i).data.Temperature_measured
    """
    try:
        from scipy.io import loadmat
    except ImportError:
        raise ImportError("scipy is required to load .mat files: pip install scipy")

    mat  = loadmat(mat_path, simplify_cells=True)
    data = mat[battery_id]
    cycles_raw = data["cycle"]

    records     = []
    cycle_count = 0
    capacities  = []

    for cyc in cycles_raw:
        if cyc["type"] != "discharge":
            continue
        cycle_count += 1
        d           = cyc["data"]
        capacity    = float(np.array(d["Capacity"]).ravel()[-1])
        voltage     = float(np.mean(np.array(d["Voltage_measured"]).ravel()))
        current     = float(np.mean(np.array(d["Current_measured"]).ravel()))
        temperature = float(np.mean(np.array(d["Temperature_measured"]).ravel()))
        capacities.append(capacity)

    # Second pass — compute RUL and SOH once we know EOL cycle
    initial_cap = capacities[0]
    eol_cycle   = next((i + 1 for i, c in enumerate(capacities)
                        if c <= EOL_THRESHOLD), len(capacities))

    for i, cap in enumerate(capacities):
        cycle   = i + 1
        soh     = float(np.clip((cap - EOL_THRESHOLD) /
                                (initial_cap - EOL_THRESHOLD), 0, 1))
        rul     = max(0, eol_cycle - cycle)
        records.append({
            "cycle": cycle, "capacity": round(cap, 4),
            "voltage": 0.0, "current": 0.0, "temperature": 0.0,
            "soh": round(soh, 4), "rul": rul, "is_eol": cap <= EOL_THRESHOLD,
        })

    return {"battery_id": battery_id, "records": records,
            "initial_capacity": initial_cap, "eol_threshold": EOL_THRESHOLD}

In [6]:

def load_battery(battery_id: str, data_dir: str = "./nasa_battery_data") -> dict:
    """Auto-select real data if available, else fall back to synthetic."""
    mat_path = os.path.join(data_dir, f"{battery_id}.mat")
    if os.path.exists(mat_path):
        print(f"  [DATA] Loading real NASA data: {mat_path}")
        return load_real_nasa_data(mat_path, battery_id)
    else:
        print(f"  [DATA] .mat not found — using synthetic data for {battery_id}")
        return generate_synthetic_nasa_data(battery_id)


In [7]:
def extract_features(records: list, window: int = WINDOW_SIZE) -> np.ndarray:
    """
    Sliding-window feature extraction over battery discharge cycles.

    Features (9 total) per sample:
        cap_mean    — mean capacity in window
        cap_std     — std of capacity in window
        cap_slope   — linear regression slope of capacity (fade rate)
        vol_mean    — mean voltage
        vol_std     — std of voltage
        temp_mean   — mean temperature
        temp_std    — std of temperature
        cap_last    — most recent capacity value
        cap_delta   — capacity drop across window (first − last)

    Returns
    -------
    X : np.ndarray  shape (N, 9)
    y : np.ndarray  shape (N,)     RUL values
    meta : list of dicts           cycle / soh info for each sample
    """
    caps   = np.array([r["capacity"]    for r in records])
    vols   = np.array([r["voltage"]     for r in records])
    temps  = np.array([r["temperature"] for r in records])
    ruls   = np.array([r["rul"]         for r in records])
    sohs   = np.array([r["soh"]         for r in records])
    cycles = np.array([r["cycle"]       for r in records])

    X, y, meta = [], [], []

    for i in range(window, len(records)):
        c_win  = caps[i - window : i]
        v_win  = vols[i - window : i]
        t_win  = temps[i - window : i]

        # Linear slope (least-squares over window index)
        x_idx  = np.arange(window, dtype=float)
        slope  = np.polyfit(x_idx, c_win, 1)[0]

        feat = [
            c_win.mean(),           # cap_mean
            c_win.std() + 1e-9,     # cap_std
            slope,                  # cap_slope
            v_win.mean(),           # vol_mean
            v_win.std() + 1e-9,     # vol_std
            t_win.mean(),           # temp_mean
            t_win.std() + 1e-9,     # temp_std
            c_win[-1],              # cap_last
            c_win[0] - c_win[-1],   # cap_delta (degradation magnitude)
        ]
        X.append(feat)
        y.append(ruls[i])
        meta.append({"cycle": cycles[i], "soh": sohs[i]})

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), meta

In [8]:
class MinMaxNormalizer:
    """Per-feature min-max scaling to [0, 1]."""

    def __init__(self):
        self.x_min = self.x_max = None
        self.y_min = self.y_max = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.x_min = X.min(axis=0)
        self.x_max = X.max(axis=0)
        self.y_min = float(y.min())
        self.y_max = float(y.max())
        return self

    def transform_X(self, X: np.ndarray) -> np.ndarray:
        rng = self.x_max - self.x_min
        rng[rng == 0] = 1.0
        return (X - self.x_min) / rng

    def transform_y(self, y: np.ndarray) -> np.ndarray:
        rng = self.y_max - self.y_min or 1.0
        return (y - self.y_min) / rng

    def inverse_y(self, y_norm: np.ndarray) -> np.ndarray:
        return y_norm * (self.y_max - self.y_min) + self.y_min

In [9]:
class DenseLayer:
    """Fully-connected layer with optional ReLU activation."""

    def __init__(self, in_dim: int, out_dim: int, activation: str = "relu"):
        # He initialisation
        scale        = math.sqrt(2.0 / in_dim)
        self.W       = np.random.randn(in_dim, out_dim).astype(np.float32) * scale
        self.b       = np.zeros(out_dim, dtype=np.float32)
        self.activation = activation
        # Adam optimiser state
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)
        # cache for backward pass
        self._input = self._z = None

    def forward(self, x: np.ndarray) -> np.ndarray:
        self._input = x
        self._z     = x @ self.W + self.b          # (batch, out_dim)
        if self.activation == "relu":
            return np.maximum(0, self._z)
        elif self.activation == "leaky_relu":
            return np.where(self._z > 0, self._z, 0.01 * self._z)
        return self._z                              # linear (output layer)

    def backward(self, grad_out: np.ndarray) -> np.ndarray:
        if self.activation == "relu":
            grad_out = grad_out * (self._z > 0)
        elif self.activation == "leaky_relu":
            grad_out = grad_out * np.where(self._z > 0, 1.0, 0.01)
        self.dW = self._input.T @ grad_out          # (in, out)
        self.db = grad_out.sum(axis=0)              # (out,)
        return grad_out @ self.W.T                  # (batch, in)

In [10]:
class NeuralNetwork:
    """
    Feedforward MLP for RUL regression.

    Architecture
    ─────────────
    Input  → Dense(64, ReLU) → Dense(32, ReLU) → Dense(16, ReLU) → Dense(1, Linear)

    Optimiser : Adam (β1=0.9, β2=0.999, ε=1e-8)
    Loss      : Mean Squared Error (MSE)
    """

    def __init__(self, input_dim: int = 9,
                 hidden_sizes: list = None,
                 activation: str = "relu"):
        if hidden_sizes is None:
            hidden_sizes = HIDDEN_SIZES

        dims   = [input_dim] + hidden_sizes
        layers = []
        for i in range(len(dims) - 1):
            layers.append(DenseLayer(dims[i], dims[i + 1], activation=activation))
        layers.append(DenseLayer(dims[-1], 1, activation="linear"))   # output
        self.layers = layers

        self.history = {"train_loss": [], "val_loss": []}
        self._t      = 0    # Adam step counter

In [15]:
def forward(self, x: np.ndarray) -> np.ndarray:
        for layer in self.layers:
            x = layer.forward(x)
        return x                        # shape (batch, 1)



In [16]:
def mse_loss(self, y_pred: np.ndarray, y_true: np.ndarray) -> float:
        return float(np.mean((y_pred.ravel() - y_true.ravel()) ** 2))

In [17]:
def backward(self, y_pred: np.ndarray, y_true: np.ndarray) -> None:
        grad = 2.0 * (y_pred - y_true.reshape(-1, 1)) / len(y_true)
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

In [18]:
def adam_step(self, lr: float = LEARNING_RATE,
                  beta1: float = 0.9, beta2: float = 0.999,
                  eps: float = 1e-8) -> None:
        self._t += 1
        for layer in self.layers:
            for p, dp, m, v in [
                (layer.W, layer.dW, layer.mW, layer.vW),
                (layer.b, layer.db, layer.mb, layer.vb),
            ]:
                m[:] = beta1 * m + (1 - beta1) * dp
                v[:] = beta2 * v + (1 - beta2) * dp ** 2
                m_hat = m / (1 - beta1 ** self._t)
                v_hat = v / (1 - beta2 ** self._t)
                p    -= lr * m_hat / (np.sqrt(v_hat) + eps)

In [19]:
def predict(self, X: np.ndarray) -> np.ndarray:
        return self.forward(X).ravel()

In [20]:
def fit(self, X_train: np.ndarray, y_train: np.ndarray,
            X_val:   np.ndarray = None, y_val:   np.ndarray = None,
            epochs: int = EPOCHS, lr: float = LEARNING_RATE,
            batch_size: int = BATCH_SIZE, verbose: int = 10) -> "NeuralNetwork":
        """
        Mini-batch stochastic gradient descent with Adam.

        Parameters
        ----------
        X_train, y_train : training data (normalised)
        X_val,   y_val   : validation data (optional)
        epochs           : number of full passes over training data
        lr               : Adam learning rate
        batch_size       : samples per gradient update
        verbose          : print every N epochs (0 = silent)
        """
        n = len(X_train)
        for epoch in range(1, epochs + 1):
            # Shuffle
            idx = np.random.permutation(n)
            X_s, y_s = X_train[idx], y_train[idx]

            # Mini-batches
            epoch_loss = 0.0
            for start in range(0, n, batch_size):
                xb = X_s[start : start + batch_size]
                yb = y_s[start : start + batch_size]
                yp = self.forward(xb)
                epoch_loss += self.mse_loss(yp, yb) * len(xb)
                self.backward(yp, yb)
                self.adam_step(lr)

            train_loss = epoch_loss / n
            self.history["train_loss"].append(train_loss)

            # Validation
            val_loss = None
            if X_val is not None:
                yp_val   = self.forward(X_val)
                val_loss = self.mse_loss(yp_val, y_val)
                self.history["val_loss"].append(val_loss)

            if verbose and epoch % verbose == 0:
                val_str = f"  Val MSE: {val_loss:.6f}" if val_loss is not None else ""
                print(f"  Epoch {epoch:>4}/{epochs}  |  Train MSE: {train_loss:.6f}{val_str}")

        return self

In [36]:

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Regression metrics standard in PHM literature.

    Returns
    -------
    dict : mae, rmse, mse, r2, mape, score (NASA scoring function)
    """
    y_true, y_pred = y_true.ravel(), y_pred.ravel()
    n        = len(y_true)
    mae      = float(np.mean(np.abs(y_true - y_pred)))
    mse      = float(np.mean((y_true - y_pred) ** 2))
    rmse     = math.sqrt(mse)
    ss_res   = np.sum((y_true - y_pred) ** 2)
    ss_tot   = np.sum((y_true - y_true.mean()) ** 2)
    r2       = 1.0 - ss_res / (ss_tot + 1e-12)
    mape     = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8)))) * 100

    # NASA prognostics scoring function
    err = y_pred - y_true
    nasa_score = float(np.sum(
        np.where(err < 0,
                 np.exp(-err / 13.0) - 1,
                 np.exp( err / 10.0) - 1)
    ))

    return {
        "n":          n,
        "mae":        mae,
        "rmse":       rmse,
        "mse":        mse,
        "r2":         r2,
        "mape":       mape,
        "nasa_score": nasa_score,
    }

In [37]:
def print_metrics(metrics: dict, title: str = "Evaluation") -> None:
    bar = "─" * 40
    print(f"\n  {bar}")
    print(f"  {title}")
    print(f"  {bar}")
    print(f"  Samples    : {metrics['n']}")
    print(f"  MAE        : {metrics['mae']:.4f}  cycles")
    print(f"  RMSE       : {metrics['rmse']:.4f}  cycles")
    print(f"  MSE        : {metrics['mse']:.6f}")
    print(f"  R²         : {metrics['r2']:.6f}")
    print(f"  MAPE       : {metrics['mape']:.2f} %")
    print(f"  NASA Score : {metrics['nasa_score']:.4f}  (lower = better)")
    print(f"  {bar}")

In [45]:
def plot_all(train_data: dict, test_data: dict,
             meta_test: list, y_test_true: np.ndarray,
             y_test_pred: np.ndarray, model: NeuralNetwork,
             metrics: dict, save_path: str = None) -> None:
    """Generates a 6-panel diagnostic figure."""

    plt.style.use("dark_background")
    fig = plt.figure(figsize=(18, 11))
    fig.patch.set_facecolor("#0b0f1a")
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

    CYAN    = "#00d4ff"
    VIOLET  = "#7c3aed"
    GREEN   = "#10b981"
    AMBER   = "#f59e0b"
    PANEL   = "#111827"

    def ax_style(ax, title, xlabel, ylabel):
        ax.set_facecolor(PANEL)
        ax.set_title(title, color="white", fontsize=11, pad=8, fontweight="bold")
        ax.set_xlabel(xlabel, color="#64748b", fontsize=8)
        ax.set_ylabel(ylabel, color="#64748b", fontsize=8)
        ax.tick_params(colors="#64748b", labelsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor("#1e293b")
        ax.grid(True, color="#1e293b", linewidth=0.5, linestyle="--")

    ax1 = fig.add_subplot(gs[0, 0])
    tr_cycles = [r["cycle"]   for r in train_data["records"]]
    tr_caps   = [r["capacity"] for r in train_data["records"]]
    te_cycles = [r["cycle"]   for r in test_data["records"]]
    te_caps   = [r["capacity"] for r in test_data["records"]]
    ax1.plot(tr_cycles, tr_caps, color=CYAN,   lw=1.5, label=train_data["battery_id"])
    ax1.plot(te_cycles, te_caps, color=VIOLET, lw=1.5, label=test_data["battery_id"])
    ax1.axhline(EOL_THRESHOLD, color=AMBER, lw=1, ls="--", label=f"EOL ({EOL_THRESHOLD} Ah)")
    ax1.legend(fontsize=7, framealpha=0.3)
    ax_style(ax1, "Capacity Fade Curves", "Cycle", "Capacity (Ah)")

    ax2   = fig.add_subplot(gs[0, 1])
    tl    = model.history["train_loss"]
    vl    = model.history["val_loss"]
    ax2.plot(tl, color=CYAN,  lw=1.5, label="Train MSE")
    ax2.plot(vl, color=AMBER, lw=1.5, ls="--", label="Val MSE")
    ax2.set_yscale("log")
    ax2.legend(fontsize=7, framealpha=0.3)
    ax_style(ax2, "Training / Validation Loss", "Epoch", "MSE (log scale)")

    ax3     = fig.add_subplot(gs[0, 2])
    t_cycs  = [m["cycle"] for m in meta_test]
    ax3.fill_between(t_cycs, y_test_true, alpha=0.2, color=GREEN)
    ax3.plot(t_cycs, y_test_true, color=GREEN,  lw=1.5, label="Actual RUL")
    ax3.plot(t_cycs, y_test_pred, color=AMBER,  lw=1.5, ls="--", label="Predicted RUL")
    ax3.legend(fontsize=7, framealpha=0.3)
    ax_style(ax3, f"RUL Prediction — {test_data['battery_id']}", "Cycle", "RUL (cycles)")

    ax5 = fig.add_subplot(gs[1, 1])
    res = y_test_pred - y_test_true
    ax5.scatter(t_cycs, res, alpha=0.45, s=12, c=AMBER, edgecolors="none")
    ax5.axhline(0, color="#64748b", lw=1, ls="--")
    ax_style(ax5, "Residuals (Predicted − Actual)", "Cycle", "Error (cycles)")

    ax6 = fig.add_subplot(gs[1, 2])
    sohs = [m["soh"] for m in meta_test]
    ax6.fill_between(t_cycs, sohs, alpha=0.25, color=CYAN)
    ax6.plot(t_cycs, sohs, color=CYAN, lw=1.5)
    ax6.axhline(0.8, color=AMBER, lw=1, ls="--", label="SOH = 0.8 (20% fade)")
    ax6.set_ylim(0, 1.05)
    ax6.legend(fontsize=7, framealpha=0.3)
    ax_style(ax6, f"State of Health — {test_data['battery_id']}", "Cycle", "SOH")

    # Metrics annotation
    txt = (f"MAE  = {metrics['mae']:.2f} cyc\n"
               f"RMSE = {metrics['rmse']:.2f} cyc\n"
               f"R²   = {metrics['r2']:.4f}\n"
               f"MAPE = {metrics['mape']:.1f} %")
    ax6.text(0.97, 0.95, txt, transform=ax6.transAxes,
    va="top", ha="right", fontsize=7.5,
    color="white", fontfamily="monospace",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#0d1525", alpha=0.85))

    # Title
    fig.suptitle(
        "NASA PCoE Battery RUL — Neural Network Prognostics Pipeline",
        color="white", fontsize=14, fontweight="bold", y=0.98
    )

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"\n  [PLOT] Saved → {save_path}")
    plt.close(fig)

In [46]:
#  6. SAVE / LOAD MODEL  (lightweight .npz checkpoint)
# ══════════════════════════════════════════════════════════════════════════════

def save_model(model: NeuralNetwork, norm: MinMaxNormalizer,
               path: str = "./outputs/rul_model.npz") -> None:
    arrays = {}
    for i, layer in enumerate(model.layers):
        arrays[f"W_{i}"] = layer.W
        arrays[f"b_{i}"] = layer.b
    arrays["x_min"]  = norm.x_min
    arrays["x_max"]  = norm.x_max
    arrays["y_min"]  = np.array([norm.y_min])
    arrays["y_max"]  = np.array([norm.y_max])
    np.savez(path, **arrays)
    print(f"  [SAVE] Model weights → {path}")

In [47]:
def load_model(path: str, input_dim: int = 9,
               hidden_sizes: list = None) -> tuple:
    if hidden_sizes is None:
        hidden_sizes = HIDDEN_SIZES
    data = np.load(path)
    model = NeuralNetwork(input_dim=input_dim, hidden_sizes=hidden_sizes)
    for i, layer in enumerate(model.layers):
        layer.W = data[f"W_{i}"]
        layer.b = data[f"b_{i}"]
    norm = MinMaxNormalizer()
    norm.x_min = data["x_min"]
    norm.x_max = data["x_max"]
    norm.y_min = float(data["y_min"])
    norm.y_max = float(data["y_max"])
    print(f"  [LOAD] Model loaded from {path}")
    return model, norm

In [48]:
#  7. MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline():
    print("\n" + "=" * 60)
    print("  NASA PCoE Battery RUL — Neural Network Pipeline")
    print("=" * 60)


In [50]:
# ── 7.1 Load data ────────────────────────────────────────────────────
print("\n[1/6] Loading battery data …")
train_data = load_battery(TRAIN_BATTERY)
test_data  = load_battery(TEST_BATTERY)
print(f"      Train: {len(train_data['records'])} discharge cycles ({TRAIN_BATTERY})")
print(f"      Test : {len(test_data['records'])} discharge cycles ({TEST_BATTERY})")


[1/6] Loading battery data …
  [DATA] .mat not found — using synthetic data for B0005
  [DATA] .mat not found — using synthetic data for B0007
      Train: 168 discharge cycles (B0005)
      Test : 168 discharge cycles (B0007)


In [52]:
# ── 7.2 Feature engineering ──────────────────────────────────────────
print("\n[2/6] Feature engineering (window = {}) …".format(WINDOW_SIZE))
X_train, y_train, meta_train = extract_features(train_data["records"], WINDOW_SIZE)
X_test,  y_test,  meta_test  = extract_features(test_data["records"],  WINDOW_SIZE)
print(f"      X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"      X_test : {X_test.shape}    y_test : {y_test.shape}")


[2/6] Feature engineering (window = 10) …
      X_train: (158, 9)   y_train: (158,)
      X_test : (158, 9)    y_test : (158,)


In [53]:
# ── 7.3 Normalise ────────────────────────────────────────────────────
print("\n[3/6] Normalising features …")
norm = MinMaxNormalizer().fit(X_train, y_train)
X_train_n = norm.transform_X(X_train)
y_train_n = norm.transform_y(y_train)
X_test_n  = norm.transform_X(X_test)


[3/6] Normalising features …


In [55]:
# Train / validation split (80 / 20)
split      = int(0.8 * len(X_train_n))
X_tr, X_vl = X_train_n[:split], X_train_n[split:]
y_tr, y_vl = y_train_n[:split], y_train_n[split:]
print(f"Train split: {len(X_tr)}  |  Val split: {len(X_vl)}")

Train split: 126  |  Val split: 32


In [57]:
# ── 7.4 Build & train model ──────────────────────────────────────────
print(f"\n[4/6] Building & training MLP …")
print(f"      Architecture : 9 → {' → '.join(map(str, HIDDEN_SIZES))} → 1")
print(f"      Epochs: {EPOCHS}  |  LR: {LEARNING_RATE}  |  Batch: {BATCH_SIZE}")
print()

# Fix: Manually attach methods to the NeuralNetwork class
# These functions are defined in separate cells but need explicit assignment
# to become methods of the NeuralNetwork class.
NeuralNetwork.forward   = forward
NeuralNetwork.mse_loss  = mse_loss
NeuralNetwork.backward  = backward
NeuralNetwork.adam_step = adam_step
NeuralNetwork.predict   = predict
NeuralNetwork.fit       = fit

model = NeuralNetwork(input_dim=9, hidden_sizes=HIDDEN_SIZES, activation="relu")
model.fit(X_tr, y_tr, X_val=X_vl, y_val=y_vl,
              epochs=EPOCHS, lr=LEARNING_RATE,
              batch_size=BATCH_SIZE, verbose=10)


[4/6] Building & training MLP …
      Architecture : 9 → 64 → 32 → 16 → 1
      Epochs: 100  |  LR: 0.002  |  Batch: 32

  Epoch   10/100  |  Train MSE: 0.013018  Val MSE: 0.057418
  Epoch   20/100  |  Train MSE: 0.004474  Val MSE: 0.032670
  Epoch   30/100  |  Train MSE: 0.002408  Val MSE: 0.027811
  Epoch   40/100  |  Train MSE: 0.001583  Val MSE: 0.025165
  Epoch   50/100  |  Train MSE: 0.001190  Val MSE: 0.023162
  Epoch   60/100  |  Train MSE: 0.000926  Val MSE: 0.020989
  Epoch   70/100  |  Train MSE: 0.000754  Val MSE: 0.018726
  Epoch   80/100  |  Train MSE: 0.000628  Val MSE: 0.017790
  Epoch   90/100  |  Train MSE: 0.000500  Val MSE: 0.016643
  Epoch  100/100  |  Train MSE: 0.000422  Val MSE: 0.015737


In [59]:
# ── 7.5 Evaluate ─────────────────────────────────────────────────────
print("\n[5/6] Evaluating on test battery …")
y_pred_n   = model.predict(X_test_n)
y_pred     = norm.inverse_y(y_pred_n)
y_pred     = np.maximum(y_pred, 0)          # RUL ≥ 0
metrics    = compute_metrics(y_test, y_pred)
print_metrics(metrics, title=f"Test Metrics — {TEST_BATTERY}")


[5/6] Evaluating on test battery …

  ────────────────────────────────────────
  Test Metrics — B0007
  ────────────────────────────────────────
  Samples    : 158
  MAE        : 7.8216  cycles
  RMSE       : 12.7394  cycles
  MSE        : 162.291458
  R²         : 0.921985
  MAPE       : 2697489000.00 %
  NASA Score : 1005.8223  (lower = better)
  ────────────────────────────────────────


In [60]:
# ── 7.5 Evaluate ─────────────────────────────────────────────────────
print("\n[5/6] Evaluating on test battery …")
y_pred_n   = model.predict(X_test_n)
y_pred     = norm.inverse_y(y_pred_n)
y_pred     = np.maximum(y_pred, 0)          # RUL ≥ 0
metrics    = compute_metrics(y_test, y_pred)
print_metrics(metrics, title=f"Test Metrics — {TEST_BATTERY}")


[5/6] Evaluating on test battery …

  ────────────────────────────────────────
  Test Metrics — B0007
  ────────────────────────────────────────
  Samples    : 158
  MAE        : 7.8216  cycles
  RMSE       : 12.7394  cycles
  MSE        : 162.291458
  R²         : 0.921985
  MAPE       : 2697489000.00 %
  NASA Score : 1005.8223  (lower = better)
  ────────────────────────────────────────


In [62]:
# ── 7.6 Save outputs ─────────────────────────────────────────────────
print("\n[6/6] Saving model, results, and plots …")
save_model(model, norm, str(OUTPUT_DIR / "rul_model.npz"))


[6/6] Saving model, results, and plots …
  [SAVE] Model weights → outputs/rul_model.npz


In [66]:
# CSV of predictions
import csv
csv_path = OUTPUT_DIR / "predictions.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["cycle", "actual_rul", "predicted_rul",
                                            "error", "soh"])
    writer.writeheader()
    for m, ya, yp in zip(meta_test, y_test, y_pred):
        writer.writerow({
            "cycle":         m["cycle"],
            "actual_rul":    round(float(ya), 2),
            "predicted_rul": round(float(yp), 2),
            "error":         round(float(yp) - float(ya), 2),
            "soh":           round(m["soh"], 4),
        })
print(f"  [CSV] Predictions → {csv_path}")

plot_all(
    train_data, test_data,
    meta_test, y_test, y_pred,
    model, metrics,
    save_path=str(OUTPUT_DIR / "rul_diagnostics.png"),
)

# The return statement has been removed as it can only be used inside a function.

  [CSV] Predictions → outputs/predictions.csv

  [PLOT] Saved → outputs/rul_diagnostics.png
